# AutoMorph — Mozania (1.773 imagens)

Roda AutoMorph nas imagens selecionadas pelo médico para comparar com as métricas do CSV.

**Fluxo:**
1. Upload do ZIP (`mozania_selected_images.zip`, ~710 MB)
2. Instala AutoMorph + dependências
3. Executa M0→M1→M2→M3
4. Download dos resultados

**⚠ ATENÇÃO:** 1.773 imagens leva ~2-4h no Colab (T4). Se o Colab desconectar, use batches (veja célula de configuração).

In [ ]:
# === CÉLULA 0: Configuração ===
# Se o Colab desconectar no meio, ajuste BATCH_START para continuar de onde parou
BATCH_SIZE = 500       # imagens por batch (0 = todas de uma vez)
BATCH_START = 0        # índice inicial (0 = começo, 500 = segundo batch, etc)
SKIP_QUALITY = False   # True = pula M1 (usa todas as imagens, não filtra por qualidade)
print(f'Configuração: batch_size={BATCH_SIZE}, batch_start={BATCH_START}, skip_quality={SKIP_QUALITY}')

In [ ]:
# === CÉLULA 1: Instalar dependências e clonar AutoMorph ===
!pip install -q efficientnet_pytorch segmentation-models-pytorch timm albumentations

import os
if not os.path.exists('/content/AutoMorph'):
    !git clone https://github.com/rmaphoh/AutoMorph.git /content/AutoMorph
    print('AutoMorph clonado')
else:
    print('AutoMorph já existe')

# Verificar GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ Sem GPU — vai rodar em CPU (muito lento)')

In [ ]:
# === CÉLULA 2: Upload do ZIP ===
from google.colab import files
import zipfile, shutil, glob

# Upload
print('Faça upload do mozania_selected_images.zip (~710 MB)...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f'Arquivo recebido: {zip_name} ({len(uploaded[zip_name])/1e6:.0f} MB)')

# Extrair
extract_dir = '/content/mozania_images'
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)

all_images = sorted(glob.glob(f'{extract_dir}/*.jpg'))
print(f'\nImagens extraídas: {len(all_images)}')
print(f'Exemplos: {[os.path.basename(f) for f in all_images[:3]]}')

In [ ]:
# === CÉLULA 3: Preparar batch de imagens para AutoMorph ===
import shutil, glob, csv

all_images = sorted(glob.glob('/content/mozania_images/*.jpg'))
total = len(all_images)

# Selecionar batch
if BATCH_SIZE > 0:
    batch_end = min(BATCH_START + BATCH_SIZE, total)
    batch_images = all_images[BATCH_START:batch_end]
    batch_label = f'batch_{BATCH_START}_{batch_end}'
    print(f'Batch: imagens {BATCH_START} a {batch_end} de {total} ({len(batch_images)} imagens)')
else:
    batch_images = all_images
    batch_label = 'all'
    print(f'Processando TODAS as {total} imagens')

# Copiar para diretório de input do AutoMorph
# AutoMorph espera: $AUTOMORPH_DATA/images/ com as imagens
input_dir = '/content/AutoMorph/images'
if os.path.exists(input_dir):
    shutil.rmtree(input_dir)
os.makedirs(input_dir)

for img_path in batch_images:
    shutil.copy2(img_path, input_dir)

print(f'Copiadas {len(batch_images)} imagens para {input_dir}')

# Criar resolution_information.csv na RAIZ do AutoMorph
# IMPORTANTE: colunas devem ser "fundus" e "res" (não "image_id"/"resolution")
# O M0 faz: resolution_list['res'][resolution_list['fundus']==image_path]
# E se não encontra, faz except:pass silenciosamente (0 imagens processadas!)
res_path = '/content/AutoMorph/resolution_information.csv'
with open(res_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['fundus', 'res'])
    for img_path in batch_images:
        name = os.path.basename(img_path)
        w.writerow([name, '11.07'])  # μm/pixel Phelcom FOV 45°

print(f'resolution_information.csv criado na raiz ({len(batch_images)} entries)')
print(f'  Colunas: fundus, res')
print(f'  Exemplo: {os.path.basename(batch_images[0])}, 11.07')

# Limpar resultados anteriores
results_dir = '/content/AutoMorph/Results'
if os.path.exists(results_dir):
    shutil.rmtree(results_dir)
    print('Results/ limpo')

# Verificação
imgs_in_dir = glob.glob(f'{input_dir}/*.jpg')
print(f'\nVerificação: {len(imgs_in_dir)} imagens em images/')
print(f'resolution_information.csv existe: {os.path.exists(res_path)}')

In [ ]:
# === CÉLULA 4: Executar AutoMorph ===
import time

os.chdir('/content/AutoMorph')

start = time.time()
print(f'Iniciando AutoMorph em {len(batch_images)} imagens...')
print(f'Início: {time.strftime("%H:%M:%S")}')
print('='*60)

if SKIP_QUALITY:
    print('⚠ SKIP_QUALITY=True: pulando M1, usando todas as imagens')
    # Rodar pipeline completo (M0 + M1 vão rodar, mas vamos sobrescrever Good_quality)
    !bash run.sh 2>&1 | tail -20
    # Copiar TODAS as M0 para Good_quality (ignorando filtro M1)
    m0_dir = 'Results/M0'
    good_dir = 'Results/M1/Good_quality'
    os.makedirs(good_dir, exist_ok=True)
    if os.path.exists(m0_dir):
        for f in glob.glob(f'{m0_dir}/*.png'):
            shutil.copy2(f, good_dir)
        n_good = len(glob.glob(f'{good_dir}/*.png'))
        print(f'Copiadas {n_good} imagens para Good_quality (skip M1)')
    # Re-rodar M2 e M3 com todas as imagens
    os.chdir('/content/AutoMorph/M2_Vessel_seg')
    !python main_vessel.py 2>&1 | tail -5
    os.chdir('/content/AutoMorph/M2_ArteryVein')
    !python main_av.py 2>&1 | tail -5
    os.chdir('/content/AutoMorph/M2_Disc_cup')
    !python main_disc.py 2>&1 | tail -5
    os.chdir('/content/AutoMorph/M3_Feature_Measuring')
    !python main_measure.py 2>&1 | tail -10
    os.chdir('/content/AutoMorph')
else:
    # Pipeline completo normal
    !bash run.sh 2>&1

elapsed = time.time() - start
print('='*60)
print(f'Fim: {time.strftime("%H:%M:%S")} — Duração: {elapsed/60:.1f} min')

# Contar resultados
for d in ['M0', 'M1/Good_quality', 'M1/Ungradable']:
    p = f'Results/{d}'
    if os.path.exists(p):
        n = len(glob.glob(f'{p}/*.png'))
        print(f'  {d}: {n} imagens')

# Verificar CSVs gerados
import pandas as pd
for csv_name in ['Macular_Features.csv', 'Disc_Features.csv', 'results_ensemble.csv', 'crop_info.csv']:
    for root, dirs, fls in os.walk('Results'):
        if csv_name in fls:
            path = os.path.join(root, csv_name)
            tmp = pd.read_csv(path)
            print(f'  {csv_name}: {len(tmp)} rows × {len(tmp.columns)} cols')

In [ ]:
# === CÉLULA 5: Coletar resultados e criar ZIP ===
import zipfile, glob, shutil, pandas as pd

output_dir = f'/content/mozania_results_{batch_label}'
os.makedirs(output_dir, exist_ok=True)

# Coletar CSVs de resultados
csv_files_found = []
for root, dirs, fls in os.walk('/content/AutoMorph/Results'):
    for f in fls:
        if f.endswith('.csv'):
            src = os.path.join(root, f)
            # Prefixar com diretório pai para evitar colisão
            parent = os.path.basename(root)
            dst_name = f'{parent}_{f}' if parent != 'Results' else f
            shutil.copy2(src, os.path.join(output_dir, dst_name))
            csv_files_found.append(dst_name)
            tmp = pd.read_csv(src)
            print(f'  {dst_name}: {len(tmp)} rows × {len(tmp.columns)} cols')

# Salvar log
with open(os.path.join(output_dir, 'run_info.txt'), 'w') as f:
    f.write(f'batch_label: {batch_label}\n')
    f.write(f'batch_start: {BATCH_START}\n')
    f.write(f'batch_size: {BATCH_SIZE}\n')
    f.write(f'images_processed: {len(batch_images)}\n')
    f.write(f'skip_quality: {SKIP_QUALITY}\n')
    f.write(f'csv_files: {csv_files_found}\n')

# Criar ZIP
zip_path = f'/content/mozania_results_{batch_label}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fls in os.walk(output_dir):
        for f in fls:
            fp = os.path.join(root, f)
            arcname = os.path.relpath(fp, output_dir)
            zf.write(fp, arcname)

print(f'\nZIP criado: {zip_path}')
print(f'Tamanho: {os.path.getsize(zip_path)/1e3:.0f} KB')

In [ ]:
# === CÉLULA 6: Download dos resultados ===
from google.colab import files
files.download(zip_path)
print(f'Download: mozania_results_{batch_label}.zip')